# 05 — Data Enrichment: Building the Country Panel

Notebooks 01–03 analysed what happened to EU energy over 20 years. Notebook 04 showed that **renewable share is predictable but energy dependency is not** — and raised the obvious follow-up: *what actually drives the pace of the transition?*

This notebook pulls two economic features from Eurostat and joins them to the country-level energy data already in the database:

| Feature | Dataset | Why it matters |
|---|---|---|
| **GDP per capita (PPS)** | `nama_10_pc` | Richer economies can invest more in renewable infrastructure |
| **Household electricity price (€/kWh)** | `nrg_pc_202` | Higher prices shift the cost calculus toward domestic renewables |

The output is a single `country_panel` table — 27 countries × ~18 years ≈ 480 rows — that notebook 06 uses for a properly featured regression.

In [9]:
import sqlite3
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import requests

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from utils.config import DB_PATH
from utils.charts import COLORS

## 1. Base panel from the existing database

Country-level renewable share and dependency rate are already in the database. We query them here rather than recalculating — same logic, same numbers, no drift.

In [10]:
conn = sqlite3.connect(DB_PATH)

panel = pd.read_sql('''
    WITH renew AS (
        SELECT year, country,
            SUM(CASE WHEN balance_type = 'Primary production'     THEN value_gwh ELSE 0 END) AS renewable_gwh,
            SUM(CASE WHEN balance_type = 'Gross available energy' THEN value_gwh ELSE 0 END) AS gae_gwh
        FROM renewables
        WHERE energy_source = 'Renewables and biofuels'
        GROUP BY year, country
    ),
    dep AS (
        SELECT year, country,
            (SUM(CASE WHEN balance_type = 'Imports' THEN value_gwh ELSE 0 END)
             - SUM(CASE WHEN balance_type = 'Exports' THEN value_gwh ELSE 0 END))
            / NULLIF(SUM(CASE WHEN balance_type = 'Gross available energy' THEN value_gwh ELSE 0 END), 0) * 100
            AS dependency_rate
        FROM energy_dependency WHERE energy_source = 'Total'
        GROUP BY year, country
    )
    SELECT r.year, r.country,
        ROUND(r.renewable_gwh / NULLIF(r.gae_gwh, 0) * 100, 2) AS renewable_share,
        ROUND(d.dependency_rate, 2)                             AS dependency_rate
    FROM renew r JOIN dep d ON r.year = d.year AND r.country = d.country
    ORDER BY r.country, r.year
''', conn)

conn.close()

EU27 = sorted(panel['country'].unique())
print(f"{panel['country'].nunique()} countries × {panel['year'].nunique()} years = {len(panel)} rows")
panel.head()

27 countries × 20 years = 540 rows


,year,country,renewable_share,dependency_rate
0,2005,Austria,100.51,71.76
1,2006,Austria,96.02,72.25
2,2007,Austria,96.72,68.48
3,2008,Austria,97.36,68.74
4,2009,Austria,95.35,65.12


## 2. Eurostat API helper

Both new datasets come from Eurostat's Statistics API, which returns SDMX-JSON — a compact format that encodes dimension positions as integers rather than repeating strings. The parser below decodes that into a plain DataFrame.

In [11]:
BASE = "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data"

def fetch_eurostat(dataset, **params):
    r = requests.get(
        f"{BASE}/{dataset}",
        params={"format": "JSON", "lang": "EN", **params},
        timeout=90,
    )
    r.raise_for_status()
    data = r.json()

    dims, sizes = data['id'], data['size']
    dim_info = data['dimension']

    pos_to_code  = {}
    pos_to_label = {}
    for dim in dims:
        cats = dim_info[dim]['category']
        inv  = {v: k for k, v in cats['index'].items()}   # position → code
        lbl  = cats.get('label', {})                       # code → label
        pos_to_code[dim]  = inv
        pos_to_label[dim] = {p: lbl.get(c, c) for p, c in inv.items()}

    strides = [1] * len(dims)
    for i in range(len(dims) - 2, -1, -1):
        strides[i] = strides[i + 1] * sizes[i + 1]

    rows = []
    for str_idx, val in data.get('value', {}).items():
        idx = int(str_idx)
        row = {}
        for i, dim in enumerate(dims):
            pos = (idx // strides[i]) % sizes[i]
            row[dim]              = pos_to_label[dim][pos]
            row[f'{dim}__code']   = pos_to_code[dim][pos]
        row['value'] = val
        rows.append(row)

    return pd.DataFrame(rows)

## 3. GDP per capita (PPS)

Purchasing Power Standard removes price-level differences between countries — it makes GDP comparable across economies of very different sizes and price levels. `B1GQ` is GDP at market prices, the standard headline measure.

In [12]:
# Inspect available dimension values and their API codes before filtering
gdp_check = fetch_eurostat("nama_10_pc", sinceTimePeriod="2022", untilTimePeriod="2022")
print(f"Shape: {gdp_check.shape}\n")
for col in [c for c in gdp_check.columns if c != 'value' and not c.endswith('__code')]:
    pairs = gdp_check[[col, f'{col}__code']].drop_duplicates().values
    print(f"{col}:")
    for label, code in sorted(pairs, key=lambda x: str(x[0])):
        print(f"  {code!r:35s} {label!r}")
    print()

Shape: (4500, 11)

freq:
  'A'                                 'Annual'

unit:
  'CLV10_EUR_HAB'                     'Chain linked volumes (2010), euro per capita'
  'CLV15_EUR_HAB'                     'Chain linked volumes (2015), euro per capita'
  'CLV20_EUR_HAB'                     'Chain linked volumes (2020), euro per capita'
  'CLV_I10_HAB'                       'Chain linked volumes, Index 2010=100, per capita'
  'CLV_I15_HAB'                       'Chain linked volumes, Index 2015=100, per capita'
  'CLV_I20_HAB'                       'Chain linked volumes, Index 2020=100, per capita'
  'CLV_PCH_PRE_HAB'                   'Chain linked volumes, percentage change on previous period, per capita'
  'CP_EUR_HAB'                        'Current prices, euro per capita'
  'CP_PPS_EU27_2020_HAB'              'Current prices, purchasing power standard (PPS, EU27 from 2020) per capita'
  'CP_NAC_HAB'                        'Current prices, units of national currency per capita'
  'PC_E

In [13]:
gdp_raw = fetch_eurostat(
    "nama_10_pc",
    unit="CP_PPS_EU27_2020_HAB",  # current prices, PPS (EU27 from 2020) per capita
    na_item="B1GQ",               # GDP at market prices
    sinceTimePeriod="2005",
    untilTimePeriod="2024",
)
print(f"Fetched {len(gdp_raw)} rows — columns: {[c for c in gdp_raw.columns if not c.endswith('__code')]}")
gdp_raw.head()

Fetched 812 rows — columns: ['freq', 'unit', 'na_item', 'geo', 'time', 'value']


,freq,freq__code,unit,unit__code,na_item,na_item__code,geo,geo__code,time,time__code,value
0,Annual,A,"Current prices, purchasing power standard (PPS...",CP_PPS_EU27_2020_HAB,Gross domestic product at market prices,B1GQ,Albania,AL,2005,2005,4926.5
1,Annual,A,"Current prices, purchasing power standard (PPS...",CP_PPS_EU27_2020_HAB,Gross domestic product at market prices,B1GQ,Albania,AL,2006,2006,5370.1
2,Annual,A,"Current prices, purchasing power standard (PPS...",CP_PPS_EU27_2020_HAB,Gross domestic product at market prices,B1GQ,Albania,AL,2007,2007,5988.6
3,Annual,A,"Current prices, purchasing power standard (PPS...",CP_PPS_EU27_2020_HAB,Gross domestic product at market prices,B1GQ,Albania,AL,2008,2008,6572.5
4,Annual,A,"Current prices, purchasing power standard (PPS...",CP_PPS_EU27_2020_HAB,Gross domestic product at market prices,B1GQ,Albania,AL,2009,2009,6817.5


In [14]:
gdp = (
    gdp_raw
    .rename(columns={'geo': 'country', 'time': 'year', 'value': 'gdp_pps'})
    [['country', 'year', 'gdp_pps']]
    .dropna(subset=['gdp_pps'])
)
gdp['year'] = gdp['year'].astype(int)
gdp = gdp[gdp['country'].isin(EU27)].reset_index(drop=True)

print(f"{gdp['country'].nunique()} countries, {gdp['year'].nunique()} years — {len(gdp)} rows")
print(f"GDP range: {gdp['gdp_pps'].min():.0f} – {gdp['gdp_pps'].max():.0f} PPS/capita")
gdp.sample(5)

27 countries, 20 years — 540 rows
GDP range: 7865 – 97690 PPS/capita


,country,year,gdp_pps
43,Bulgaria,2008,10942.7
73,Cyprus,2018,27303.3
398,Malta,2023,41860.0
166,Greece,2011,19146.0
70,Cyprus,2015,22563.5


## 4. Household electricity prices (€/kWh)

Eurostat records electricity prices **twice a year** (S1 = Jan–Jun, S2 = Jul–Dec). We fetch both halves and average them to get one annual figure per country.

We first fetch a single year to inspect the dimension labels the API returns, then apply the right filters on the full pull.

In [15]:
# Inspect electricity prices dataset (nrg_pc_204 = household electricity)
elec_sample = fetch_eurostat("nrg_pc_204", sinceTimePeriod="2022", untilTimePeriod="2022")
print(f"nrg_pc_204 — {len(elec_sample)} rows\n")
for col in [c for c in elec_sample.columns if c != 'value' and not c.endswith('__code')]:
    pairs = elec_sample[[col, f'{col}__code']].drop_duplicates().values
    print(f"{col}:")
    for label, code in sorted(pairs, key=lambda x: str(x[0])):
        print(f"  {code!r:30s} {label!r}")
    print()

nrg_pc_204 — 4167 rows

freq:
  'S'                            'Half-yearly, semesterly'

siec:
  'E7000'                        'Electricity'

nrg_cons:
  'KWH_GE15000'                  'Consumption for 15 000 kWh or over - band DE'
  'KWH1000-2499'                 'Consumption from 1 000 kWh to 2 499 kWh - band DB'
  'KWH2500-4999'                 'Consumption from 2 500 kWh to 4 999 kWh - band DC'
  'KWH5000-14999'                'Consumption from 5 000 kWh to 14 999 kWh - band DD'
  'KWH_LT1000'                   'Consumption less than 1 000 kWh - band DA'
  'TOT_KWH'                      'Consumption of kWh - all bands'

unit:
  'KWH'                          'Kilowatt-hour'

tax:
  'I_TAX'                        'All taxes and levies included'
  'X_VAT'                        'Excluding VAT and other recoverable taxes and levies'
  'X_TAX'                        'Excluding taxes and levies'

currency:
  'EUR'                          'Euro'
  'NAC'                          'Natio

In [16]:
# Band DC (2 500–4 999 kWh/year) — EU reference band for a typical household.
# Prices all-taxes-included — what households actually pay.
elec_raw = fetch_eurostat(
    "nrg_pc_204",
    nrg_cons="KWH2500-4999",  # band DC
    unit="KWH",               # EUR per kWh
    currency="EUR",
    tax="I_TAX",              # all taxes included
    sinceTimePeriod="2005",
    untilTimePeriod="2024",
)
print(f"Fetched {len(elec_raw)} rows")
elec_raw.head()

Fetched 1368 rows


,freq,freq__code,siec,siec__code,nrg_cons,nrg_cons__code,unit,unit__code,tax,tax__code,currency,currency__code,geo,geo__code,time,time__code,value
0,"Half-yearly, semesterly",S,Electricity,E7000,Consumption from 2 500 kWh to 4 999 kWh - band DC,KWH2500-4999,Kilowatt-hour,KWH,All taxes and levies included,I_TAX,Euro,EUR,Albania,AL,2011-S1,2011-S1,0.1152
1,"Half-yearly, semesterly",S,Electricity,E7000,Consumption from 2 500 kWh to 4 999 kWh - band DC,KWH2500-4999,Kilowatt-hour,KWH,All taxes and levies included,I_TAX,Euro,EUR,Albania,AL,2011-S2,2011-S2,0.1157
2,"Half-yearly, semesterly",S,Electricity,E7000,Consumption from 2 500 kWh to 4 999 kWh - band DC,KWH2500-4999,Kilowatt-hour,KWH,All taxes and levies included,I_TAX,Euro,EUR,Albania,AL,2012-S1,2012-S1,0.1163
3,"Half-yearly, semesterly",S,Electricity,E7000,Consumption from 2 500 kWh to 4 999 kWh - band DC,KWH2500-4999,Kilowatt-hour,KWH,All taxes and levies included,I_TAX,Euro,EUR,Albania,AL,2012-S2,2012-S2,0.1167
4,"Half-yearly, semesterly",S,Electricity,E7000,Consumption from 2 500 kWh to 4 999 kWh - band DC,KWH2500-4999,Kilowatt-hour,KWH,All taxes and levies included,I_TAX,Euro,EUR,Albania,AL,2013-S1,2013-S1,0.1156


In [17]:
elec = (
    elec_raw
    .rename(columns={'geo': 'country', 'time': 'period', 'value': 'price'})
    [['country', 'period', 'price']]
    .dropna(subset=['price'])
)
elec['year'] = elec['period'].str[:4].astype(int)
elec = (
    elec[elec['country'].isin(EU27)]
    .groupby(['country', 'year'])['price']
    .mean()
    .reset_index()
    .rename(columns={'price': 'elec_price_eur_kwh'})
)

print(f"{elec['country'].nunique()} countries, {elec['year'].nunique()} years — {len(elec)} rows")
print(f"Price range: {elec['elec_price_eur_kwh'].min():.4f} – {elec['elec_price_eur_kwh'].max():.4f} EUR/kWh")
elec.sample(5)

27 countries, 18 years — 485 rows
Price range: 0.0721 – 0.5215 EUR/kWh


,country,year,elec_price_eur_kwh
438,Slovenia,2014,0.1631
307,Luxembourg,2009,0.1882
102,Czechia,2019,0.1759
428,Slovakia,2022,0.1840
477,Sweden,2017,0.1984


## 5. Join and check coverage

A left join keeps every country-year in the base panel and marks gaps in the new columns as NaN — making missing data visible rather than silently dropping rows.

In [18]:
enriched = (
    panel
    .merge(gdp,  on=['country', 'year'], how='left')
    .merge(elec, on=['country', 'year'], how='left')
)
enriched['t'] = enriched['year'] - enriched['year'].min()

missing = enriched[['gdp_pps', 'elec_price_eur_kwh']].isnull().sum()
print("Missing values per feature:")
print(missing.to_string())
print(f"\nTotal rows: {len(enriched)}  |  Complete rows: {enriched.dropna().shape[0]}")
enriched.head()

Missing values per feature:
gdp_pps                0
elec_price_eur_kwh    55

Total rows: 540  |  Complete rows: 485


,year,country,renewable_share,dependency_rate,gdp_pps,elec_price_eur_kwh,t
0,2005,Austria,100.51,71.76,28495.1,NaN,0
1,2006,Austria,96.02,72.25,29734.7,NaN,1
2,2007,Austria,96.72,68.48,30945.9,0.17400,2
3,2008,Austria,97.36,68.74,31853.9,0.17755,3
4,2009,Austria,95.35,65.12,30722.4,0.19090,4


In [19]:
panel_clean = enriched.dropna(subset=['gdp_pps', 'elec_price_eur_kwh']).reset_index(drop=True)
years = f"{panel_clean['year'].min()}–{panel_clean['year'].max()}"
print(f"Final panel: {panel_clean['country'].nunique()} countries × {years} = {len(panel_clean)} rows")
panel_clean.describe().round(2)

Final panel: 27 countries × 2007–2024 = 485 rows


,year,renewable_share,dependency_rate,gdp_pps,elec_price_eur_kwh,t
count,485.00,485.00,485.00,485.00,485.00,485.00
mean,2015.52,98.25,57.01,29781.10,0.18,10.52
std,5.18,16.63,24.48,13982.81,0.07,5.18
min,2007.00,55.09,-24.25,9972.40,0.07,2.00
25%,2011.00,92.29,39.51,20778.30,0.14,6.00
50%,2016.00,98.55,58.40,27390.80,0.17,11.00
75%,2020.00,102.70,75.20,34447.60,0.22,15.00
max,2024.00,164.39,104.14,97690.20,0.52,19.00


In [20]:
# ── Sanity checks ────────────────────────────────────────────────────────────
ok = True

# 1. Electricity price range — EUR/kWh should sit between ~0.05 and ~0.50
p_min, p_max = panel_clean["elec_price_eur_kwh"].min(), panel_clean["elec_price_eur_kwh"].max()
if 0.04 < p_min and p_max < 0.60:
    print(f"PASS Electricity price range: {p_min:.4f} - {p_max:.4f} EUR/kWh  (looks like kWh prices)")
else:
    print(f"FAIL Electricity price range looks wrong: {p_min:.4f} - {p_max:.4f}  (expected 0.05-0.50 EUR/kWh)")
    ok = False

# 2. Germany 2022 electricity spiked to ~0.35+ EUR/kWh during the energy crisis
de22 = panel_clean.query("country == 'Germany' and year == 2022")["elec_price_eur_kwh"]
if not de22.empty and de22.iloc[0] > 0.30:
    print(f"PASS Germany 2022 electricity: {de22.iloc[0]:.4f} EUR/kWh  (energy crisis spike confirmed)")
else:
    val = de22.iloc[0] if not de22.empty else "missing"
    print(f"FAIL Germany 2022 electricity: {val}  (expected >0.30 EUR/kWh)")
    ok = False

# 3. GDP range — PPS per capita should be ~5 000-100 000 for EU27
g_min, g_max = panel_clean["gdp_pps"].min(), panel_clean["gdp_pps"].max()
if 3_000 < g_min and g_max < 150_000:
    print(f"PASS GDP range: {g_min:,.0f} - {g_max:,.0f} PPS/capita")
else:
    print(f"FAIL GDP range looks wrong: {g_min:,.0f} - {g_max:,.0f}  (expected ~5 000-100 000)")
    ok = False

# 4. Luxembourg should have one of the highest GDPs
lu_gdp = panel_clean.query("country == 'Luxembourg' and year == 2022")["gdp_pps"]
eu_median = panel_clean.query("year == 2022")["gdp_pps"].median()
if not lu_gdp.empty and lu_gdp.iloc[0] > eu_median * 2:
    print(f"PASS Luxembourg 2022 GDP: {lu_gdp.iloc[0]:,.0f} PPS  (well above EU median {eu_median:,.0f})")
else:
    print(f"FAIL Luxembourg 2022 GDP: {lu_gdp.values}  EU median: {eu_median:,.0f}")
    ok = False

# 5. No duplicate country-year rows
dupes = panel_clean.duplicated(subset=["country", "year"]).sum()
if dupes == 0:
    print(f"PASS No duplicate country-year rows")
else:
    print(f"FAIL {dupes} duplicate country-year rows found")
    ok = False

# 6. Sweden renewable share should be among the highest (hydro + wind)
se_share = panel_clean.query("country == 'Sweden' and year == 2022")["renewable_share"]
if not se_share.empty and se_share.iloc[0] > 40:
    print(f"PASS Sweden 2022 renewable share: {se_share.iloc[0]:.1f}%  (hydro-heavy, as expected)")
else:
    print(f"FAIL Sweden 2022 renewable share: {se_share.values}  (expected >40%)")
    ok = False

print(f"{'All checks passed.' if ok else 'Some checks failed — review above.'}")

PASS Electricity price range: 0.0721 - 0.5215 EUR/kWh  (looks like kWh prices)
PASS Germany 2022 electricity: 0.3318 EUR/kWh  (energy crisis spike confirmed)
PASS GDP range: 9,972 - 97,690 PPS/capita
PASS Luxembourg 2022 GDP: 90,395 PPS  (well above EU median 32,420)
PASS No duplicate country-year rows
PASS Sweden 2022 renewable share: 94.4%  (hydro-heavy, as expected)
All checks passed.


## 6. Does the signal exist?

A model can only surface signal that exists in the raw data. Before notebook 06 models anything, a visual check: do the new features show any relationship with renewable share across the panel?

In [21]:
fig = go.Figure()
for country in panel_clean['country'].unique():
    c = panel_clean[panel_clean['country'] == country]
    fig.add_trace(go.Scatter(
        x=c['gdp_pps'], y=c['renewable_share'],
        mode='markers',
        marker=dict(color=COLORS['renewables'], size=5, opacity=0.45),
        showlegend=False,
        hovertemplate=f"<b>{country}</b><br>GDP: %{{x:,.0f}} PPS<br>Renewable share: %{{y:.1f}}%<extra></extra>",
    ))
fig.update_layout(
    title='GDP per capita vs Renewable share — all country-years',
    xaxis_title='GDP per capita (PPS)', yaxis_title='Renewable share (%)',
    plot_bgcolor='white',
    yaxis=dict(gridcolor='#eeeeee'),
    xaxis=dict(gridcolor='#eeeeee'),
)
fig.show()

In [22]:
fig = go.Figure()
for country in panel_clean['country'].unique():
    c = panel_clean[panel_clean['country'] == country]
    fig.add_trace(go.Scatter(
        x=c['elec_price_eur_kwh'], y=c['renewable_share'],
        mode='markers',
        marker=dict(color=COLORS['neutral'], size=5, opacity=0.45),
        showlegend=False,
        hovertemplate=f"<b>{country}</b><br>Price: €%{{x:.3f}}/kWh<br>Renewable share: %{{y:.1f}}%<extra></extra>",
    ))
fig.update_layout(
    title='Electricity price vs Renewable share — all country-years',
    xaxis_title='Household electricity price (€/kWh)', yaxis_title='Renewable share (%)',
    plot_bgcolor='white',
    yaxis=dict(gridcolor='#eeeeee'),
    xaxis=dict(gridcolor='#eeeeee'),
)
fig.show()

In [23]:
corr_cols = ['renewable_share', 'dependency_rate', 'gdp_pps', 'elec_price_eur_kwh', 't']
corr = panel_clean[corr_cols].corr().round(2)

fig = go.Figure(go.Heatmap(
    z=corr.values,
    x=corr.columns.tolist(),
    y=corr.index.tolist(),
    colorscale='RdBu', zmid=0, zmin=-1, zmax=1,
    text=corr.values.round(2),
    texttemplate='%{text}',
    showscale=True,
))
fig.update_layout(
    title='Feature correlation matrix',
    plot_bgcolor='white',
    width=600, height=500,
)
fig.show()

## 7. Store to database

The enriched panel goes into the same `energy.db` as the rest of the project. Notebook 06 reads directly from here — no files to pass around, no version drift between notebooks.

In [24]:
conn = sqlite3.connect(DB_PATH)
panel_clean.to_sql('country_panel', conn, if_exists='replace', index=False)
conn.close()

conn = sqlite3.connect(DB_PATH)
verify = pd.read_sql(
    "SELECT COUNT(*) as rows, COUNT(DISTINCT country) as countries, "
    "MIN(year) as yr_min, MAX(year) as yr_max FROM country_panel",
    conn
)
conn.close()

print("country_panel stored:")
display(verify)

country_panel stored:


,rows,countries,yr_min,yr_max
0,485,27,2007,2024
